In [ ]:
"""
Batch segmentation inference and paired-mask merging.

Main functions
--------------
1. Run sliding-window segmentation on large images.
2. Infer processing size from physical size encoded in file/folder names.
3. Preserve the original aspect ratio when resizing outputs.
4. Extract defect instances from predicted masks.
5. Automatically merge paired *_0.png and *_1.png segmentation results.
6. Save segmentation masks, instance visualizations, merged marker maps and CSV summaries.
"""

import os
import re
import sys
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

from glob import glob
from PIL import Image
from collections import deque
from scipy.signal import convolve2d
from monai.inferers import SlidingWindowInferer
from torch import nn

sys.path.append(os.path.join(os.getcwd(), "unet"))
from train_pt import SegModel


# =============================================================================
# Global configuration
# =============================================================================

ROI_SIZE = 256

CKPT_PATH = (
    r"G:\Moire_Code\seg\logs\ReS2_Unetplus_ResNet34_mask1\version_0"
    r"\checkpoints\epoch=89-val_loss=0.1312-val_dice=0.7729.ckpt"
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "unet++"
IN_CHANNELS = 1

# Physical calibration: 256 pixels correspond to this physical length in nm.
PHYSICAL_SIZE_PER_ROI = 4.5

# Instance extraction mode:
# - "connected": connected-component analysis
# - "convolution": convolution-based clustering
EXTRACTION_MODE = "connected"

# Connected-component extraction parameters.
MIN_INSTANCE_PIXELS = 10
MAX_INSTANCE_PIXELS = None

# Convolution-based extraction parameters.
CONV_NUM = 1
CONV_THRESH = 0.5
CENTER_RADIUS = 20
MIN_PIXELS = 30

# Marker-rendering parameters.
MERGE_CIRCLE_RADIUS = 6
MERGE_MIN_PIXELS = 0

# Marker colors in BGRA format.
COLOR_MASK1 = (227, 190, 148, 255)
COLOR_MASK2 = (128, 133, 232, 255)


# =============================================================================
# Physical-size utilities
# =============================================================================

def parse_physical_size_from_path(path):
    """
    Parse physical size from file or parent-folder name.

    Supported name pattern
    ----------------------
    12.5x8.0
    12.5X8.0
    12.5×8.0

    Returns
    -------
    tuple[float, float] or None
        (width_nm, height_nm)
    """
    basename = os.path.basename(path)
    parent_name = os.path.basename(os.path.dirname(path))

    pattern = r"(\d+\.?\d*)\s*[xX×]\s*(\d+\.?\d*)"

    match = re.search(pattern, basename)
    if match is None:
        match = re.search(pattern, parent_name)

    if match:
        return float(match.group(1)), float(match.group(2))

    return None


def calculate_target_pixel_size(
    physical_size_nm,
    physical_per_roi=PHYSICAL_SIZE_PER_ROI,
    roi_pixels=ROI_SIZE,
):
    """
    Convert physical size in nm into target processing size in pixels.

    Returns
    -------
    tuple[int, int]
        (height_pixels, width_pixels)
    """
    width_nm, height_nm = physical_size_nm
    pixels_per_nm = roi_pixels / physical_per_roi

    target_width = int(round(width_nm * pixels_per_nm))
    target_height = int(round(height_nm * pixels_per_nm))

    return target_height, target_width


def calculate_output_size(original_size, processing_size):
    """
    Compute output size while preserving the original aspect ratio.

    Returns
    -------
    tuple
        ((output_height, output_width), scale_factor)
    """
    orig_h, orig_w = original_size
    proc_h, proc_w = processing_size

    scale_h = proc_h / orig_h
    scale_w = proc_w / orig_w
    scale = max(scale_h, scale_w)

    output_h = int(round(orig_h * scale))
    output_w = int(round(orig_w * scale))

    return (output_h, output_w), scale


# =============================================================================
# Instance extraction
# =============================================================================

def extract_instances(mask_img, mode="connected", **kwargs):
    """
    Extract instance centers from a binary segmentation mask.

    Parameters
    ----------
    mask_img : ndarray
        Segmentation mask with values in {0, 1} or {0, 255}.

    mode : str
        "connected" or "convolution".

    Returns
    -------
    centers : list[tuple[int, int]]
        Instance centers in (row, col) format.

    stats : list[dict]
        Instance statistics. Only connected mode returns detailed stats.
    """
    if mask_img.max() > 1:
        binary = (mask_img > 127).astype(np.uint8)
    else:
        binary = (mask_img > 0).astype(np.uint8)

    if mode == "connected":
        return _extract_connected(binary, **kwargs)

    if mode == "convolution":
        return _extract_convolution(binary, **kwargs)

    raise ValueError(f"Unknown extraction mode: {mode}")


def _extract_connected(binary, min_pixels=1, max_pixels=None):
    """
    Extract instances using connected-component analysis.
    """
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary)

    instance_centers = []
    instance_stats = []

    for label_id in range(1, num_labels):
        area = stats[label_id, cv2.CC_STAT_AREA]

        if area < min_pixels:
            continue
        if max_pixels is not None and area > max_pixels:
            continue

        cx, cy = centroids[label_id]
        center = (int(round(cy)), int(round(cx)))

        instance_centers.append(center)
        instance_stats.append({
            "center": center,
            "area": area,
            "bbox": (
                stats[label_id, cv2.CC_STAT_LEFT],
                stats[label_id, cv2.CC_STAT_TOP],
                stats[label_id, cv2.CC_STAT_WIDTH],
                stats[label_id, cv2.CC_STAT_HEIGHT],
            ),
        })

    return instance_centers, instance_stats


def _extract_convolution(binary, num_convs=1, thresh=0.5, radius=20):
    """
    Extract instances using convolution-based clustering.
    """
    conv_mask = convolve(num_convs, binary, thresh)
    centers = get_center_list(conv_mask, radius)

    return centers, []


def filter_small_instances(mask_img, min_pixels=20):
    """
    Remove connected components smaller than min_pixels.
    """
    binary = (mask_img > 127).astype(np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary)

    filtered = np.zeros_like(mask_img)

    for label_id in range(1, num_labels):
        if stats[label_id, cv2.CC_STAT_AREA] >= min_pixels:
            filtered[labels == label_id] = 255

    return filtered


def convolve(num_convs, mask_img, thresh=0.5):
    """
    Smooth and threshold a binary mask using repeated 3 x 3 convolution.
    """
    if mask_img.ndim != 2:
        raise ValueError(f"Mask must be a 2D array, got ndim={mask_img.ndim}")

    ptp_val = np.ptp(mask_img)

    if ptp_val > 0:
        conv_img = (mask_img - np.min(mask_img)) / ptp_val
    else:
        conv_img = np.zeros_like(mask_img, dtype=np.float32)

    conv_img = (conv_img >= thresh).astype(np.int32)
    kernel = np.ones((3, 3), dtype=np.float32) / 9.0

    for _ in range(num_convs):
        conv_img = convolve2d(conv_img, kernel, mode="same")
        conv_img = (conv_img >= thresh).astype(np.int32)

    return conv_img


def get_center_list(conv_mask, radius):
    """
    Cluster positive pixels and return one center for each cluster.
    """
    i_indices, j_indices = np.where(conv_mask == 1)
    center_list = list(zip(i_indices, j_indices))

    if not center_list:
        return []

    remaining = deque(center_list)
    instance_centers = []

    while remaining:
        i, j = remaining.popleft()
        cluster = [(i, j)]
        temp_remaining = []

        while remaining:
            ik, jk = remaining.popleft()
            dist_sq = (i - ik) ** 2 + (j - jk) ** 2

            if dist_sq < radius ** 2:
                cluster.append((ik, jk))
            else:
                temp_remaining.append((ik, jk))

        remaining.extend(temp_remaining)

        cluster_np = np.array(cluster)
        center_i = int(round(np.mean(cluster_np[:, 0])))
        center_j = int(round(np.mean(cluster_np[:, 1])))

        instance_centers.append((center_i, center_j))

    return instance_centers


def scale_centers(centers, from_size, to_size):
    """
    Scale center coordinates from one image size to another.
    """
    if not centers:
        return []

    scale_h = to_size[0] / from_size[0]
    scale_w = to_size[1] / from_size[1]

    scaled_centers = []

    for ci, cj in centers:
        new_ci = int(round(ci * scale_h))
        new_cj = int(round(cj * scale_w))

        new_ci = min(max(0, new_ci), to_size[0] - 1)
        new_cj = min(max(0, new_cj), to_size[1] - 1)

        scaled_centers.append((new_ci, new_cj))

    return scaled_centers


# =============================================================================
# Debug utilities
# =============================================================================

def analyze_instance_sizes(mask_path_or_array):
    """
    Analyze the connected-component area distribution in a mask.
    """
    if isinstance(mask_path_or_array, str):
        mask = cv2.imread(mask_path_or_array, cv2.IMREAD_GRAYSCALE)
    else:
        mask = mask_path_or_array

    if mask is None:
        print("Failed to read mask")
        return []

    binary = (mask > 127).astype(np.uint8) if mask.max() > 1 else mask.astype(np.uint8)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary)

    areas = [stats[i, cv2.CC_STAT_AREA] for i in range(1, num_labels)]

    if areas:
        print("Instance analysis:")
        print(f"  Count: {len(areas)}")
        print(f"  Area range: {min(areas)} ~ {max(areas)} pixels")
        print(f"  Mean area: {np.mean(areas):.1f} pixels")
        print(f"  Smallest areas: {sorted(areas)[:20]}{'...' if len(areas) > 20 else ''}")
    else:
        print("No instances detected")

    return areas


def debug_instance_extraction(mask_path, save_path=None):
    """
    Visualize intermediate steps of instance extraction.
    """
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if mask is None:
        print(f"Failed to read: {mask_path}")
        return

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    axes[0, 0].imshow(mask, cmap="gray")
    axes[0, 0].set_title("1. Raw mask")
    axes[0, 0].axis("off")

    centers_conn, stats_conn = extract_instances(
        mask,
        mode="connected",
        min_pixels=MIN_INSTANCE_PIXELS,
        max_pixels=MAX_INSTANCE_PIXELS,
    )

    axes[0, 1].imshow(mask, cmap="gray")
    for cy, cx in centers_conn:
        axes[0, 1].scatter(cx, cy, c="red", s=30, marker="x")
    axes[0, 1].set_title(f"2. Connected mode\nmin={MIN_INSTANCE_PIXELS}, count={len(centers_conn)}")
    axes[0, 1].axis("off")

    centers_conv, _ = extract_instances(
        mask,
        mode="convolution",
        num_convs=CONV_NUM,
        thresh=CONV_THRESH,
        radius=CENTER_RADIUS,
    )

    axes[0, 2].imshow(mask, cmap="gray")
    for cy, cx in centers_conv:
        axes[0, 2].scatter(cx, cy, c="blue", s=30, marker="o")
    axes[0, 2].set_title(f"3. Convolution mode\nconv={CONV_NUM}, count={len(centers_conv)}")
    axes[0, 2].axis("off")

    filtered = filter_small_instances(mask, MIN_INSTANCE_PIXELS)
    axes[1, 0].imshow(filtered, cmap="gray")
    axes[1, 0].set_title(f"4. Filtered mask\nmin_pixels={MIN_INSTANCE_PIXELS}")
    axes[1, 0].axis("off")

    conv_result = convolve(CONV_NUM, mask, CONV_THRESH)
    axes[1, 1].imshow(conv_result, cmap="gray")
    axes[1, 1].set_title(f"5. Convolved mask\nn={CONV_NUM}, thresh={CONV_THRESH}")
    axes[1, 1].axis("off")

    areas = analyze_instance_sizes(mask)

    if areas:
        axes[1, 2].hist(areas, bins=min(50, len(areas)), edgecolor="black")
        axes[1, 2].axvline(
            x=MIN_INSTANCE_PIXELS,
            color="r",
            linestyle="--",
            label=f"min={MIN_INSTANCE_PIXELS}",
        )
        axes[1, 2].set_xlabel("Area (pixels)")
        axes[1, 2].set_ylabel("Count")
        axes[1, 2].set_title("6. Instance area distribution")
        axes[1, 2].legend()
    else:
        axes[1, 2].text(0.5, 0.5, "No instance", ha="center", va="center")
        axes[1, 2].set_title("6. Instance area distribution")

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Debug figure saved to: {save_path}")

    plt.show()
    plt.close()


# =============================================================================
# Mask merging and marker rendering
# =============================================================================

def merge_mask_pair(
    mask1_path,
    mask2_path,
    output_path,
    circle_radius=MERGE_CIRCLE_RADIUS,
    min_pixels=MERGE_MIN_PIXELS,
    extraction_mode=EXTRACTION_MODE,
):
    """
    Merge two paired segmentation masks into one transparent BGRA marker image.
    """
    mask1 = cv2.imread(mask1_path, cv2.IMREAD_GRAYSCALE)
    mask2 = cv2.imread(mask2_path, cv2.IMREAD_GRAYSCALE)

    if mask1 is None:
        raise FileNotFoundError(f"Failed to read mask1: {mask1_path}")
    if mask2 is None:
        raise FileNotFoundError(f"Failed to read mask2: {mask2_path}")

    h, w = mask1.shape

    if mask2.shape != mask1.shape:
        mask2 = cv2.resize(mask2, (w, h), interpolation=cv2.INTER_NEAREST)

    if extraction_mode == "connected":
        centers1, _ = extract_instances(
            mask1,
            mode="connected",
            min_pixels=MIN_INSTANCE_PIXELS,
            max_pixels=MAX_INSTANCE_PIXELS,
        )
        centers2, _ = extract_instances(
            mask2,
            mode="connected",
            min_pixels=MIN_INSTANCE_PIXELS,
            max_pixels=MAX_INSTANCE_PIXELS,
        )
    else:
        centers1, _ = extract_instances(
            mask1,
            mode="convolution",
            num_convs=CONV_NUM,
            thresh=CONV_THRESH,
            radius=CENTER_RADIUS,
        )
        centers2, _ = extract_instances(
            mask2,
            mode="convolution",
            num_convs=CONV_NUM,
            thresh=CONV_THRESH,
            radius=CENTER_RADIUS,
        )

    canvas = np.zeros((h, w, 4), dtype=np.uint8)

    for cy, cx in centers1:
        cv2.circle(canvas, (cx, cy), circle_radius, COLOR_MASK1, -1)

    for cy, cx in centers2:
        cv2.circle(canvas, (cx, cy), circle_radius, COLOR_MASK2, -1)

    cv2.imwrite(output_path, canvas)

    return {
        "mask1_instances": len(centers1),
        "mask2_instances": len(centers2),
        "output_path": output_path,
    }


def save_single_marker(
    mask_path,
    output_path,
    color,
    circle_radius=MERGE_CIRCLE_RADIUS,
    extraction_mode=EXTRACTION_MODE,
):
    """
    Save one transparent marker image from a single segmentation mask.
    """
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if mask is None:
        raise FileNotFoundError(f"Failed to read mask: {mask_path}")

    h, w = mask.shape

    if extraction_mode == "connected":
        centers, _ = extract_instances(
            mask,
            mode="connected",
            min_pixels=MIN_INSTANCE_PIXELS,
            max_pixels=MAX_INSTANCE_PIXELS,
        )
    else:
        centers, _ = extract_instances(
            mask,
            mode="convolution",
            num_convs=CONV_NUM,
            thresh=CONV_THRESH,
            radius=CENTER_RADIUS,
        )

    canvas = np.zeros((h, w, 4), dtype=np.uint8)

    for cy, cx in centers:
        cv2.circle(canvas, (cx, cy), circle_radius, color, -1)

    cv2.imwrite(output_path, canvas)

    return len(centers)


# =============================================================================
# Model loading
# =============================================================================

def load_model(ckpt_path):
    """
    Load the segmentation model and checkpoint for inference.
    """
    from core.model import SMPModelFactory

    class InferenceModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.model = SMPModelFactory(in_channels=IN_CHANNELS).get_model(MODEL_NAME)

        def forward(self, x):
            return self.model(x)

    model = InferenceModel()

    checkpoint = torch.load(ckpt_path, map_location=DEVICE)

    if "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])

        epoch = checkpoint.get("epoch", "N/A")
        val_dice = checkpoint.get("metrics", {}).get("val_dice", "N/A")

        if isinstance(val_dice, float):
            print(f"Checkpoint loaded: epoch={epoch}, val_dice={val_dice:.4f}")
        else:
            print(f"Checkpoint loaded: epoch={epoch}, val_dice={val_dice}")
    else:
        model.load_state_dict(checkpoint)

    model.eval()
    model.to(DEVICE)

    print(f"Model: {MODEL_NAME}, input channels: {IN_CHANNELS}, device: {DEVICE}")
    print(f"Instance extraction: {EXTRACTION_MODE}, min_pixels: {MIN_INSTANCE_PIXELS}")

    return model


# =============================================================================
# Single-image inference
# =============================================================================

def seg_single_image(
    model,
    image_path,
    save_dir="./results/",
    target_size=None,
    auto_resize=True,
    enable_instance_vis=True,
):
    """
    Segment one image and save the mask, optional instance visualization and metadata.
    """
    mask_save_dir = os.path.join(save_dir, "masks")
    os.makedirs(mask_save_dir, exist_ok=True)

    instance_vis_dir = os.path.join(save_dir, "instance_vis") if enable_instance_vis else None
    if enable_instance_vis:
        os.makedirs(instance_vis_dir, exist_ok=True)

    image = cv2.imread(image_path)

    if image is None:
        raise FileNotFoundError(f"Failed to load image: {image_path}")

    image_gray_original = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    original_size = image_gray_original.shape[:2]

    physical_size = None
    processing_size = None
    output_size = original_size
    scale_factor = 1.0

    if auto_resize and target_size is None:
        physical_size = parse_physical_size_from_path(image_path)

        if physical_size:
            processing_size = calculate_target_pixel_size(physical_size)
            output_size, scale_factor = calculate_output_size(original_size, processing_size)

            print(f"Physical size: {physical_size[0]:.2f} x {physical_size[1]:.2f} nm")
            print(f"Original size: {original_size[1]} x {original_size[0]} px")
            print(f"Processing size: {processing_size[1]} x {processing_size[0]} px")
            print(f"Output size: {output_size[1]} x {output_size[0]} px, scale={scale_factor:.2f}")
        else:
            print("Physical size not found. Use original image size.")
            processing_size = original_size

    elif target_size is not None:
        processing_size = target_size
        output_size, scale_factor = calculate_output_size(original_size, processing_size)

    else:
        processing_size = original_size

    if processing_size != original_size:
        image_gray_processed = cv2.resize(
            image_gray_original,
            (processing_size[1], processing_size[0]),
            interpolation=cv2.INTER_LINEAR,
        )
    else:
        image_gray_processed = image_gray_original.copy()

    image_norm = image_gray_processed.astype(np.float32) / 255.0

    image_tensor = torch.from_numpy(image_norm).float()
    image_tensor = image_tensor.unsqueeze(0).unsqueeze(0).to(DEVICE)

    inferer = SlidingWindowInferer(
        roi_size=(ROI_SIZE, ROI_SIZE),
        sw_batch_size=1,
        overlap=0.25,
        mode="gaussian",
        padding_mode="reflect",
    )

    model.eval()

    with torch.no_grad():
        output = inferer(image_tensor, model)
        pred = torch.argmax(output, dim=1).cpu().numpy()[0]

    if EXTRACTION_MODE == "connected":
        centers_processing, instance_stats = extract_instances(
            pred,
            mode="connected",
            min_pixels=MIN_INSTANCE_PIXELS,
            max_pixels=MAX_INSTANCE_PIXELS,
        )

        if instance_stats:
            areas = [s["area"] for s in instance_stats]
            print(f"Instance area: min={min(areas)}, max={max(areas)}, mean={np.mean(areas):.1f}")

    else:
        centers_processing, _ = extract_instances(
            pred,
            mode="convolution",
            num_convs=CONV_NUM,
            thresh=CONV_THRESH,
            radius=CENTER_RADIUS,
        )

    pred_output = cv2.resize(
        pred.astype(np.uint8),
        (output_size[1], output_size[0]),
        interpolation=cv2.INTER_NEAREST,
    )

    centers_output = scale_centers(
        centers_processing,
        processing_size,
        output_size,
    )

    pred_vis = (pred_output * 255).astype(np.uint8)
    base_name = os.path.splitext(os.path.basename(image_path))[0]

    mask_save_path = os.path.join(mask_save_dir, f"{base_name}_seg.png")
    Image.fromarray(pred_vis).save(mask_save_path)

    instance_vis_path = None

    if enable_instance_vis:
        image_gray_output = cv2.resize(
            image_gray_original,
            (output_size[1], output_size[0]),
            interpolation=cv2.INTER_LINEAR,
        )

        plt.figure(figsize=(15, 7))

        plt.subplot(121)
        plt.imshow(image_gray_output, cmap="gray")
        plt.title(f"Image\n{output_size[1]} x {output_size[0]} px")
        plt.axis("off")

        plt.subplot(122)
        plt.imshow(pred_vis, cmap="viridis")

        if centers_output:
            marker_size = max(20, int(80 * scale_factor))
            for idx, (ci, cj) in enumerate(centers_output):
                plt.scatter(
                    cj,
                    ci,
                    color="red",
                    s=marker_size,
                    marker="x",
                    label="Instance" if idx == 0 else "",
                )
            plt.legend(loc="upper right")

        title_str = f"Segmentation: {len(centers_output)} instances"

        if physical_size:
            title_str += f"\n{physical_size[0]:.2f} x {physical_size[1]:.2f} nm"

        title_str += f"\n{EXTRACTION_MODE} mode, min_px={MIN_INSTANCE_PIXELS}"

        plt.title(title_str)
        plt.axis("off")

        instance_vis_path = os.path.join(instance_vis_dir, f"{base_name}_instance.png")

        plt.tight_layout()
        plt.savefig(instance_vis_path, dpi=300, bbox_inches="tight")
        plt.close()

    result = {
        "image_path": image_path,
        "mask_path": mask_save_path,
        "instance_vis_path": instance_vis_path,
        "instance_count": len(centers_output),
        "physical_size": physical_size,
        "original_size": original_size,
        "processing_size": processing_size,
        "output_size": output_size,
        "scale_factor": scale_factor,
    }

    print(f"{base_name}: instances={len(centers_output)}")

    return result


# =============================================================================
# Batch processing
# =============================================================================

def batch_process_folder(
    root_dir,
    save_dir="./results_batch/",
    auto_resize=True,
    enable_instance_vis=True,
    enable_merge=True,
    file_suffixes=("_0.png", "_1.png"),
    model=None,
):
    """
    Process all subfolders and optionally merge paired *_0.png and *_1.png results.
    """
    if model is None:
        model = load_model(CKPT_PATH)

    subfolders = [
        folder
        for folder in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, folder))
    ]

    print(f"\n{'=' * 70}")
    print("Batch processing started")
    print(f"Root directory: {root_dir}")
    print(f"Subfolders: {len(subfolders)}")
    print(f"Auto resize: {auto_resize}")
    print(f"Auto merge: {enable_merge}")
    print(f"Instance extraction: {EXTRACTION_MODE}")
    print(f"Min instance pixels: {MIN_INSTANCE_PIXELS}")
    print(f"{'=' * 70}\n")

    all_results = []
    merge_results = []
    total_images = 0
    total_instances = 0

    for idx, subfolder in enumerate(subfolders):
        subfolder_path = os.path.join(root_dir, subfolder)

        print(f"\n[{idx + 1}/{len(subfolders)}] {subfolder}")
        print("-" * 60)

        image_files = []

        for suffix in file_suffixes:
            pattern = os.path.join(subfolder_path, f"*{suffix}")
            image_files.extend(glob(pattern))

        if not image_files:
            print("No matched image files found")
            continue

        subfolder_save_dir = os.path.join(save_dir, subfolder)
        subfolder_results = {}

        for img_path in sorted(image_files):
            try:
                result = seg_single_image(
                    model=model,
                    image_path=img_path,
                    save_dir=subfolder_save_dir,
                    auto_resize=auto_resize,
                    enable_instance_vis=enable_instance_vis,
                )

                all_results.append(result)
                total_images += 1
                total_instances += result["instance_count"]

                subfolder_results[os.path.basename(img_path)] = result

            except Exception as exc:
                print(f"Failed: {os.path.basename(img_path)} - {exc}")

        if enable_merge:
            merge_result = auto_merge_pairs(subfolder_results, subfolder_save_dir, subfolder)
            if merge_result:
                merge_results.append(merge_result)

    print(f"\n{'=' * 70}")
    print("Batch processing finished")
    print(f"Processed images: {total_images}")
    print(f"Total instances: {total_instances}")
    print(f"Merged pairs: {len(merge_results)}")
    print(f"Save directory: {save_dir}")
    print(f"{'=' * 70}\n")

    save_summary_csv(all_results, merge_results, save_dir)

    return all_results, merge_results


def auto_merge_pairs(subfolder_results, save_dir, subfolder_name):
    """
    Find paired *_0.png and *_1.png results and merge their segmentation masks.
    """
    pairs = {}

    for filename in subfolder_results.keys():
        if "_0.png" in filename:
            base = filename.replace("_0.png", "")
            pairs.setdefault(base, {})["_0"] = subfolder_results[filename]
        elif "_1.png" in filename:
            base = filename.replace("_1.png", "")
            pairs.setdefault(base, {})["_1"] = subfolder_results[filename]

    merge_results = []

    for base_name, pair in pairs.items():
        if "_0" not in pair or "_1" not in pair:
            continue

        mask1_path = pair["_0"]["mask_path"]
        mask2_path = pair["_1"]["mask_path"]

        merge_dir = os.path.join(save_dir, "merged")
        os.makedirs(merge_dir, exist_ok=True)

        output_path = os.path.join(merge_dir, f"{base_name}_merged.png")

        try:
            result = merge_mask_pair(mask1_path, mask2_path, output_path)

            print(f"Merge completed: {base_name}")
            print(f"  Mask0 instances: {result['mask1_instances']}")
            print(f"  Mask1 instances: {result['mask2_instances']}")

            marker0_path = os.path.join(merge_dir, f"{base_name}_0_marker.png")
            marker1_path = os.path.join(merge_dir, f"{base_name}_1_marker.png")

            cnt0 = save_single_marker(
                mask1_path,
                marker0_path,
                color=COLOR_MASK1,
                circle_radius=MERGE_CIRCLE_RADIUS,
                extraction_mode=EXTRACTION_MODE,
            )

            cnt1 = save_single_marker(
                mask2_path,
                marker1_path,
                color=COLOR_MASK2,
                circle_radius=MERGE_CIRCLE_RADIUS,
                extraction_mode=EXTRACTION_MODE,
            )

            print(f"  Marker _0: {cnt0} instances -> {os.path.basename(marker0_path)}")
            print(f"  Marker _1: {cnt1} instances -> {os.path.basename(marker1_path)}")

            merge_results.append({
                "subfolder": subfolder_name,
                "base_name": base_name,
                "mask0_instances": result["mask1_instances"],
                "mask1_instances": result["mask2_instances"],
                "merged_path": output_path,
                "physical_size": pair["_0"].get("physical_size"),
                "output_size": pair["_0"].get("output_size"),
                "scale_factor": pair["_0"].get("scale_factor"),
            })

        except Exception as exc:
            print(f"Merge failed: {base_name} - {exc}")

    return merge_results if merge_results else None


def save_summary_csv(results, merge_results, save_dir):
    """
    Save segmentation and merge summaries as CSV files.
    """
    import csv

    os.makedirs(save_dir, exist_ok=True)

    seg_csv_path = os.path.join(save_dir, "segmentation_summary.csv")

    with open(seg_csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        writer.writerow([
            "image_path",
            "physical_size_nm",
            "original_pixels",
            "processing_pixels",
            "output_pixels",
            "scale_factor",
            "instance_count",
            "extraction_mode",
        ])

        for item in results:
            physical_str = (
                f"{item['physical_size'][0]:.2f}x{item['physical_size'][1]:.2f}"
                if item["physical_size"] else "N/A"
            )
            original_str = (
                f"{item['original_size'][1]}x{item['original_size'][0]}"
                if item.get("original_size") else "N/A"
            )
            processing_str = (
                f"{item['processing_size'][1]}x{item['processing_size'][0]}"
                if item.get("processing_size") else "N/A"
            )
            output_str = (
                f"{item['output_size'][1]}x{item['output_size'][0]}"
                if item.get("output_size") else "N/A"
            )
            scale_str = f"{item.get('scale_factor', 1.0):.2f}x"

            writer.writerow([
                os.path.basename(item["image_path"]),
                physical_str,
                original_str,
                processing_str,
                output_str,
                scale_str,
                item["instance_count"],
                EXTRACTION_MODE,
            ])

    print(f"Segmentation summary saved to: {seg_csv_path}")

    if not merge_results:
        return

    flat_merge = []

    for item in merge_results:
        if isinstance(item, list):
            flat_merge.extend(item)
        elif item:
            flat_merge.append(item)

    if not flat_merge:
        return

    merge_csv_path = os.path.join(save_dir, "merge_summary.csv")

    with open(merge_csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        writer.writerow([
            "subfolder",
            "base_name",
            "physical_size_nm",
            "output_pixels",
            "scale_factor",
            "mask0_instances",
            "mask1_instances",
        ])

        for item in flat_merge:
            physical_str = (
                f"{item['physical_size'][0]:.2f}x{item['physical_size'][1]:.2f}"
                if item.get("physical_size") else "N/A"
            )
            output_str = (
                f"{item['output_size'][1]}x{item['output_size'][0]}"
                if item.get("output_size") else "N/A"
            )
            scale_str = f"{item.get('scale_factor', 1.0):.2f}x"

            writer.writerow([
                item["subfolder"],
                item["base_name"],
                physical_str,
                output_str,
                scale_str,
                item["mask0_instances"],
                item["mask1_instances"],
            ])

    print(f"Merge summary saved to: {merge_csv_path}")


# =============================================================================
# Main entry
# =============================================================================

if __name__ == "__main__":
    model = load_model(CKPT_PATH)

    batch_root_dir = (
        r"xx"
        r"xx"
    )
    batch_save_dir = (
        r"xx"
        r"xx"
    )

    results, merge_results = batch_process_folder(
        root_dir=batch_root_dir,
        save_dir=batch_save_dir,
        auto_resize=True,
        enable_instance_vis=True,
        enable_merge=True,
        file_suffixes=("_0.png", "_1.png"),
        model=model,
    )

    # Optional debug example:
    # debug_instance_extraction(r"path\to\mask.png", save_path="debug.png")